<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_11_Exact_ADNI_Crosswalk_Athena_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import os, sys, json, csv, re, hashlib, urllib.request, io, zipfile, platform
from datetime import datetime, timezone
from collections import defaultdict, Counter
from decimal import Decimal, InvalidOperation
from html.parser import HTMLParser
import pandas as pd

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

BASE=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT=Path(os.environ.get('ICHI_PROJECT_DIR', str(BASE/'drive/MyDrive/NeuroFHIR_ICHI2027')))
OUT=PROJECT/'phase11_outputs'
OUT.mkdir(parents=True,exist_ok=True)
PRIVATE=BASE/'ICHI2027_Phase11_PRIVATE_SESSION'
PRIVATE.mkdir(parents=True,exist_ok=True)
REPORT=OUT/'ICHI2027_Phase11_SHAREABLE.json'
SOURCE_NAME='TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'
PRIOR_NAME='ICHI2027_Phase10_SHAREABLE.json'
PUBLIC_URL='https://adni.bitbucket.io/reference/ucsffsl51.html'
NOW=datetime.now(timezone.utc)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):
            h.update(b)
    return h.hexdigest()

def canonical_decimal(value):
    try: x=Decimal(str(value).strip())
    except InvalidOperation: return None
    return x if x.is_finite() else None

def locate(name,dirs):
    return next((d/name for d in dirs if (d/name).is_file()),None)

ROOTS=[BASE,BASE/'inputs',PROJECT/'inputs',PROJECT/'source_data',
       PROJECT/'source_metadata',PROJECT/'trace_outputs',PROJECT/'vocabulary']

EXPECTED={'RID','PTID'}|{
    f'{tp}_{x}' for tp in ('BL','M12') for x in (
        'VISCODE','EXAMDATE','IMAGEUID','LONIUID','VERSION','OVERALLQC','TEMPQC','VENTQC',
        'LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV')
}


Mounted at /content/drive


In [2]:
prior_path=locate(PRIOR_NAME,[PROJECT/'phase10_outputs',BASE,BASE/'inputs'])
if prior_path is None and 'google.colab' in sys.modules:
    from google.colab import files
    upload=files.upload()
    if PRIOR_NAME in upload:
        prior_path=PRIVATE/PRIOR_NAME
        prior_path.write_bytes(upload[PRIOR_NAME])
if prior_path is None:
    raise FileNotFoundError(PRIOR_NAME)
prior=json.loads(prior_path.read_text(encoding='utf-8'))
assert prior['phase']=='10_provenance_trace_and_athena_evidence_recovery'
assert prior['gates']['phase09_actual_fhir_candidate_etl_confirmed'] is True
PRIOR_SHA=sha256_file(prior_path)


In [3]:
source_path=locate(SOURCE_NAME,[BASE,PROJECT/'source_data',PROJECT/'inputs',BASE/'inputs'])
if source_path is None and 'google.colab' in sys.modules:
    from google.colab import files
    upload=files.upload()
    if SOURCE_NAME in upload:
        source_path=PRIVATE/SOURCE_NAME
        source_path.write_bytes(upload[SOURCE_NAME])
if source_path is None:
    raise FileNotFoundError(SOURCE_NAME)

source=pd.read_csv(source_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
source.columns=[x.strip() for x in source.columns]
missing=sorted(EXPECTED-set(source.columns))
if missing: raise ValueError('Missing actual source fields: '+repr(missing))
if len(source)!=258: raise ValueError(f'Unexpected source row count {len(source)} at {source_path}; stop and re-audit')
source['RID']=source.RID.str.strip()
if source.RID.eq('').any() or source.RID.duplicated().any():
    raise ValueError('Invalid RID integrity; stop and re-audit')
if source.PTID.str.strip().eq('').any(): raise ValueError('Missing PTID; stop')

private_manifest={
    'run_utc':NOW.isoformat(),
    'source_file_sha256':sha256_file(source_path),
    'source_columns':source.columns.tolist(),
    'phase10_shareable_sha256':PRIOR_SHA,
}
(PRIVATE/'source_manifest_PRIVATE.json').write_text(json.dumps(private_manifest,indent=2))
source_profile={
    'rows':int(len(source)),
    'columns':int(len(source.columns)),
    'validated_source_schema':True,
    'rid_unique':bool(source.RID.is_unique),
    'metadata_present_by_timepoint':{
        tp:{key:int(source[f'{tp}_{key}'].str.strip().ne('').sum())
            for key in ('EXAMDATE','IMAGEUID','LONIUID','VERSION','OVERALLQC','TEMPQC','VENTQC')}
        for tp in ('BL','M12')
    },
}


In [4]:
class VisibleText(HTMLParser):
    def __init__(self):
        super().__init__();self.parts=[];self.ignored=0
    def handle_starttag(self,tag,attrs):
        if tag in {'script','style','noscript','svg'}:self.ignored+=1
        if tag in {'p','div','li','tr','br','h1','h2','h3'}:self.parts.append('\n')
    def handle_endtag(self,tag):
        if tag in {'script','style','noscript','svg'}:self.ignored=max(0,self.ignored-1)
        if tag in {'p','div','li','tr','br','h1','h2','h3'}:self.parts.append('\n')
    def handle_data(self,data):
        if not self.ignored:self.parts.append(data+' ')

DOC_FIELD_CANDIDATES={
    'LHIPPOC':'ST29SV','RHIPPOC':'ST88SV',
    'LENTORHIN':'ST24CV','RENTORHIN':'ST83CV',
    'LMIDTEMP':'ST40CV','RMIDTEMP':'ST99CV',
    'ICV':'ST10CV',
}
public_ref={
    'url':PUBLIC_URL,'retrieved':False,'document_sha256':None,
    'reference_table_identifier':'UCSFFSL51',
    'source_feature_candidates':{},
    'source_identifier_semantics':{},
    'not_a_local_export_attestation':True,
}
try:
    req=urllib.request.Request(PUBLIC_URL,headers={'User-Agent':'Mozilla/5.0 (scientific-reproducibility-audit)'})
    with urllib.request.urlopen(req,timeout=35) as resp: body=resp.read()
    parser=VisibleText();parser.feed(body.decode('utf-8',errors='replace'))
    text='\n'.join(' '.join(line.split()) for line in ''.join(parser.parts).splitlines() if line.strip())
    public_ref['retrieved']=True
    public_ref['document_sha256']=hashlib.sha256(body).hexdigest()
    public_ref['source_identifier_semantics']={
      'LONISID':('LONI study UID' if re.search(r'\bLONISID\b[^\n]{0,100}LONI study UID',text,re.I) else None),
      'LONIUID':('LONI series UID' if re.search(r'\bLONIUID\b[^\n]{0,100}LONI series UID',text,re.I) else None),
      'IMAGEUID':('image identifier' if re.search(r'\bIMAGEUID\b[^\n]{0,100}image processed',text,re.I) else None),
    }
    for feature,code in DOC_FIELD_CANDIDATES.items():
        m=re.search(r'\b'+re.escape(code)+r'\b([^\n]{0,190})',text,re.I)
        body_text=m.group(1).strip() if m else None
        unit_match=re.search(r'\bmm\s*3\b|\bmm³\b',body_text or '',re.I)
        public_ref['source_feature_candidates'][feature]={
          'source_raw_field_candidate':code,
          'public_text_match':bool(m),
          'public_unit_candidate':'mm3' if unit_match else None,
          'documented_text_excerpt':(body_text[:160] if body_text else None),
          'verified_to_equal_local_export':False,
        }
    public_ref['ventricles_aggregate_definition_verified']=False
except Exception as e:
    public_ref['error_type']=type(e).__name__
    for feat,code in DOC_FIELD_CANDIDATES.items():
        public_ref['source_feature_candidates'][feat]={
          'source_raw_field_candidate':code,'public_text_match':False,
          'public_unit_candidate':None,'verified_to_equal_local_export':False}


In [5]:
metadata_dirs=[PROJECT/'source_metadata',PROJECT/'source_docs',PROJECT/'inputs'/'metadata',BASE/'source_metadata']
raw_candidates=[]
for directory in metadata_dirs:
    if directory.is_dir():
        raw_candidates.extend(p for p in directory.rglob('*')
            if p.is_file() and p.suffix.lower() in {'.csv','.tsv'}
            and p.stat().st_size < 350_000_000
            and any(s in p.name.lower() for s in ('ucsffsl','longitudinal','freesurfer','raw_export')))
raw_candidates=sorted(set(raw_candidates),key=lambda x:x.stat().st_mtime,reverse=True)
raw_profiles=[]
raw_source=None
raw_path=None
for p in raw_candidates[:25]:
    try:
        delimiter='\t' if p.suffix.lower()=='.tsv' else ','
        with open(p,'r',encoding='utf-8-sig',errors='replace',newline='') as f:
            cols=[c.strip() for c in next(csv.reader(f,delimiter=delimiter))]
        profile={'has_exact_join_keys':{'RID','EXAMDATE','IMAGEUID','LONIUID'}.issubset(cols),
                 'document_candidate_columns_present':sorted(set(DOC_FIELD_CANDIDATES.values())&set(cols)),
                 'has_LONISID':'LONISID' in cols,
                 'file_sha256_private_saved':False}
        raw_profiles.append(profile)
        if raw_path is None and profile['has_exact_join_keys']:
            raw_path=p
    except Exception:
        raw_profiles.append({'read_failed':True})
if raw_path is not None:
    delimiter='\t' if raw_path.suffix.lower()=='.tsv' else ','
    raw_source=pd.read_csv(raw_path,dtype=str,keep_default_na=False,encoding='utf-8-sig',sep=delimiter,low_memory=False)
    raw_source.columns=[c.strip() for c in raw_source.columns]
    private_manifest['optional_original_raw_export_sha256']=sha256_file(raw_path)
    (PRIVATE/'source_manifest_PRIVATE.json').write_text(json.dumps(private_manifest,indent=2))


In [6]:
join_summary={
 'original_raw_table_present':bool(raw_source is not None),
 'candidate_original_raw_tables_scanned':len(raw_profiles),
 'exact_join_key_coverage':None,
 'ambiguous_raw_keys':None,
 'source_measurement_reconciliation':{},
 'original_loni_study_id_available_for_exact_matches':False,
 'dicom_study_instance_uid_verified':False,
 'dicom_series_instance_uid_verified':False,
 'unit_claims_auto_approved':False,
}
if raw_source is not None:
    raw_source['RID']=raw_source.RID.str.strip()
    def clean_date(s):
        return pd.to_datetime(s,errors='coerce').dt.strftime('%Y-%m-%d').fillna('')
    raw_source['_EXAMDATE']=clean_date(raw_source.EXAMDATE)
    for k in ('IMAGEUID','LONIUID'):
        raw_source[k]=raw_source[k].str.strip()
    key_cols=['RID','_EXAMDATE','IMAGEUID','LONIUID']
    duplicate_keys=raw_source.duplicated(subset=key_cols,keep=False)
    join_summary['ambiguous_raw_keys']=int(duplicate_keys.sum())
    unambiguous_raw=raw_source.loc[~duplicate_keys].set_index(key_cols,drop=False)
    matches=[]
    for tp in ('BL','M12'):
        frame=source[['RID',f'{tp}_EXAMDATE',f'{tp}_IMAGEUID',f'{tp}_LONIUID']].copy()
        frame.columns=['RID','_EXAMDATE','IMAGEUID','LONIUID']
        frame['_EXAMDATE']=clean_date(frame['_EXAMDATE'])
        for k in ('IMAGEUID','LONIUID'):frame[k]=frame[k].str.strip()
        idx=pd.MultiIndex.from_frame(frame[key_cols])
        raw_matches=unambiguous_raw.reindex(idx)
        matched=raw_matches.RID.notna().to_numpy()
        matches.append((tp,raw_matches,matched))
    join_summary['exact_join_key_coverage']={tp:int(m.sum()) for tp,_,m in matches}
    if 'LONISID' in raw_source.columns:
        join_summary['original_loni_study_id_available_for_exact_matches']=any(
            bool(r.loc[m,'LONISID'].astype(str).str.strip().ne('').any()) for _,r,m in matches)
    for field,original in DOC_FIELD_CANDIDATES.items():
        results={'raw_field':original,'original_column_present':original in raw_source.columns,
                 'compared_event_pairs':0,'exact_decimal_matches':0,'mismatches':0,
                 'local_export_unit_approved':False}
        if original in raw_source.columns:
            for tp,ref,matched in matches:
                local_vals=source[f'{tp}_{field}'].tolist()
                remote_vals=ref[original].tolist()
                for i,ok in enumerate(matched):
                    if not ok:continue
                    local=canonical_decimal(local_vals[i]);raw=canonical_decimal(remote_vals[i])
                    if local is None or raw is None:continue
                    results['compared_event_pairs']+=1
                    if local==raw:results['exact_decimal_matches']+=1
                    else:results['mismatches']+=1
        join_summary['source_measurement_reconciliation'][field]=results


In [7]:
trace_dirs=[PROJECT/'trace_outputs',PROJECT/'source_metadata',PROJECT/'_private_reproducibility',BASE/'trace_outputs']
trace_files=[]
for folder in trace_dirs:
    if folder.is_dir():
        trace_files.extend(p for p in folder.rglob('*') if p.is_file()
            and p.suffix.lower() in {'.csv','.tsv','.parquet'}
            and any(x in p.name.lower() for x in ('trace','reliability','calibration')))
trace_files=sorted(set(trace_files))[:50]
trace_summary={'candidate_files':len(trace_files),'files_with_state_and_join_key':0,
               'files_with_state_rule_reference_and_join_key':0,
               'original_state_provenance_approved':False,
               'candidate_headers_aggregate':[]}
for p in trace_files:
    try:
        if p.suffix.lower()=='.parquet':
            import pyarrow.parquet as pq
            cols=pq.ParquetFile(p).schema.names
        else:
            with open(p,encoding='utf-8-sig',errors='replace',newline='') as f:
                row=next(csv.reader(f,delimiter='\t' if p.suffix.lower()=='.tsv' else ','))
            cols=[c.strip() for c in row]
        keys={x.lower() for x in cols}
        has_join_key=bool(keys & {'rid','ptid','participant_id'})
        has_state=any(re.search(r'(trace|reliab).*?(state|status)|(state|status).*?(trace|reliab)',x,re.I) for x in cols)
        has_rule=any('rule' in x.lower() and ('version' in x.lower() or 'id' in x.lower()) for x in cols)
        has_ref=any(('reference' in x.lower() or 'calibration' in x.lower()) for x in cols)
        if has_join_key and has_state: trace_summary['files_with_state_and_join_key']+=1
        if has_join_key and has_state and has_rule and has_ref:
            trace_summary['files_with_state_rule_reference_and_join_key']+=1
        trace_summary['candidate_headers_aggregate'].append({'column_count':len(cols),
           'join_key_present':has_join_key,'state_present':has_state,'rule_present':has_rule,
           'calibration_reference_present':has_ref})
    except Exception:
        trace_summary['candidate_headers_aggregate'].append({'read_error':True})


In [8]:
VOC_DIR=PROJECT/'vocabulary'
REQUIRED_VOCAB={'CONCEPT.CSV','VOCABULARY.CSV','CONCEPT_RELATIONSHIP.CSV'}
athena_packages=[]
if VOC_DIR.is_dir():
    athena_packages=sorted(p for p in VOC_DIR.rglob('*.zip') if p.is_file() and p.stat().st_size>0)
athena={'snapshot_found':False,'archive_sha256':None,'required_files_present':False,
        'concept_rows_scanned':0,'standard_valid_candidate_counts':{},
        'vocabulary_versions_in_package':0,'candidate_mappings_auto_approved':False}
for p in athena_packages:
    with zipfile.ZipFile(p) as z:
        names={Path(s).name.upper():s for s in z.namelist() if not s.endswith('/')}
        if not REQUIRED_VOCAB.issubset(names):continue
        athena['snapshot_found']=True
        athena['required_files_present']=True
        athena['archive_sha256']=sha256_file(p)
        concepts={k:0 for k in ('magnetic_resonance','hippocampus','entorhinal','temporal','ventricles','intracranial','cubic_millimeter')}
        patterns={
            'magnetic_resonance':r'magnetic resonance|\bmri\b',
            'hippocampus':r'hippocamp',
            'entorhinal':r'entorhinal',
            'temporal':r'middle temporal',
            'ventricles':r'ventric',
            'intracranial':r'intracranial',
            'cubic_millimeter':r'cubic millimet|millimeter cubed|mm\^?3',
        }
        candidates=[]
        with z.open(names['CONCEPT.CSV']) as binary:
            f=io.TextIOWrapper(binary,encoding='utf-8-sig',errors='replace',newline='')
            rdr=csv.DictReader(f,delimiter='\t')
            required={'concept_id','concept_name','domain_id','vocabulary_id','standard_concept','invalid_reason'}
            if not required.issubset(rdr.fieldnames or []):
                raise ValueError('Athena CONCEPT.csv header is inconsistent with full snapshot')
            for row in rdr:
                athena['concept_rows_scanned']+=1
                if row['standard_concept'].strip()!='S' or row['invalid_reason'].strip():continue
                for family,pat in patterns.items():
                    if re.search(pat,row['concept_name'],re.I):
                        concepts[family]+=1
                        if concepts[family] <= 35:
                            candidates.append({'family':family,'concept_id':row['concept_id'],
                                'concept_name':row['concept_name'],'vocabulary_id':row['vocabulary_id'],
                                'domain_id':row['domain_id'],'valid_start_date':row.get('valid_start_date'),
                                'valid_end_date':row.get('valid_end_date')})
        athena['standard_valid_candidate_counts']=concepts
        with z.open(names['VOCABULARY.CSV']) as binary:
            reader=csv.DictReader(io.TextIOWrapper(binary,encoding='utf-8-sig',errors='replace',newline=''),delimiter='\t')
            versions=[dict(row) for row in reader]
            athena['vocabulary_versions_in_package']=len(versions)
        (PRIVATE/'athena_candidates_PRIVATE.json').write_text(json.dumps({
            'package_sha256':athena['archive_sha256'],'candidates_not_approved':candidates,
            'vocabulary_versions':versions},indent=2))
        break


In [9]:
report={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'11_exact_ADNI_export_crosswalk_and_athena_gate',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'status':'REAL_SOURCE_CROSSWALK_AND_VOCAB_AUDIT_ONLY_NO_OFFICIAL_MICDM_LOAD_OR_FOUR_ARM_COMPARISON',
 'privacy':'Only aggregate checks, public field definitions and package hash metadata; no participant rows or identifiers.',
 'phase10_shareable_sha256':PRIOR_SHA,
 'source_profile':source_profile,
 'public_adni_source_reference':public_ref,
 'optional_original_raw_export_crosscheck':join_summary,
 'original_trace_sidecar':trace_summary,
 'athena':athena,
 'gates':{
    'real_longitudinal_source_read_and_schema_verified':True,
    'public_source_dictionary_retrieved':public_ref['retrieved'],
    'exact_original_raw_export_reconciled':bool(join_summary['original_raw_table_present'] and
       join_summary.get('exact_join_key_coverage') and
       all(c==len(source) for c in join_summary['exact_join_key_coverage'].values()) and
       all(join_summary['source_measurement_reconciliation'].get(f,{}).get('compared_event_pairs')==2*len(source) and
           join_summary['source_measurement_reconciliation'].get(f,{}).get('mismatches')==0
           for f in DOC_FIELD_CANDIDATES)),
    'measurement_unit_source_export_approved':False,
    'dicom_study_and_series_UIDs_verified':False,
    'original_TRACE_state_rule_and_reference_verified':False,
    'athena_full_snapshot_loaded':bool(athena['snapshot_found']),
    'OMOP_concept_mappings_approved':False,
    'official_MICDM_load_completed':False,
    'real_four_arm_comparison_completed':False,
 },
 'comparative_performance_scores':None,
 'MI_CDM_target_metrics':None,
}
if not join_summary['original_raw_table_present']:
    report['next_required_evidence']='EXACT_ORIGINAL_ADNI_SOURCE_EXPORT_OR_DICTIONARY_PLUS_ATHENA_AND_ORIGINAL_TRACE_SIDECAR'
elif not athena['snapshot_found']:
    report['next_required_evidence']='FULL_ATHENA_VOCABULARY_PACKAGE_AND_ORIGINAL_TRACE_SIDECAR_WITH_PROVENANCE'
else:
    report['next_required_evidence']='MANUAL_REVIEW_AND_APPROVAL_OF_EXACT_SOURCE_UNITS_UICS_AND_VOCABULARY_CROSSWALK_BEFORE_REAL_MI_CDM'
REPORT.write_text(json.dumps(report,indent=2))
print(json.dumps({'phase':report['phase'],'source_rows':source_profile['rows'],
 'original_raw_export_present':join_summary['original_raw_table_present'],
 'athena_snapshot_present':athena['snapshot_found'],
 'original_trace_evidence_candidates':trace_summary['files_with_state_rule_reference_and_join_key'],
 'next_required_evidence':report['next_required_evidence']},indent=2))


{
  "phase": "11_exact_ADNI_export_crosswalk_and_athena_gate",
  "source_rows": 258,
  "original_raw_export_present": false,
  "athena_snapshot_present": false,
  "original_trace_evidence_candidates": 0,
  "next_required_evidence": "EXACT_ORIGINAL_ADNI_SOURCE_EXPORT_OR_DICTIONARY_PLUS_ATHENA_AND_ORIGINAL_TRACE_SIDECAR"
}


In [10]:
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(REPORT))
else:
    print(str(REPORT))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>